# 07 — Bronze: PNCP (Contratos)

## Purpose
Ingests raw contract data from the PNCP landing zone into the Bronze layer.
Reads monthly JSON files, flattens all nested STRUCTs, applies technical columns,
captures schema drift via `_rescued_data`, and writes partitioned Parquet.

## Input
- `data/raw/pncp/{YYYY-MM}.json` — one JSON file per month
- 4 months available (2026-01 → 2026-04) — ~644k records total

## Output
- `data/bronze/pncp_contratos/_year_month={YYYY-MM}/data.parquet` — partitioned by month
- Schema: 55 source columns (all STRING, flattened STRUCTs) + `_rescued_data` + 5 technical
- `db/schema_drift_log.json` — drift events (non-null _rescued_data)

## Steps
1. Imports and configuration
2. Schema definition
3. Process months
4. Validation
5. Ops registration

## Notes
- All source columns cast to STRING — no type casting at Bronze (ADR-002)
- STRUCTs flattened with underscore separator: `orgaoEntidade.cnpj` → `orgaoEntidade_cnpj`
- `orgaoSubRogado.*` and `unidadeSubRogada.*` kept despite ~100% null — Bronze never drops
- `dataAssinatura`, `dataVigenciaInicio`, `dataVigenciaFim` inferred as DATE — cast to STRING
- `_rescued_data`: captures unexpected columns as JSON blob — pipeline never stops
- Idempotent — existing partitions overwritten on re-run
- No UF filter at Bronze — national scope. Filter applied at Silver (ADR-004 revised)
- ADRs: ADR-002 (STRING for CNPJ), ADR-004 (UF filter in Silver, not Bronze)


## Step 1 — Imports and configuration

In [ ]:
import json
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "utils"))
from paths import get_project_root, to_sql_path
from duckdb_utils import get_connection
from validation import CheckSuite
from bootstrap_log import append_log, make_entry
from pipeline import check_landing_gate

# ---------------------------------------------------------------------------
# Project root and paths
# ---------------------------------------------------------------------------
PROJECT_ROOT = get_project_root()

RAW_ROOT    = PROJECT_ROOT / "data" / "raw"    / "pncp"
BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze" / "pncp_contratos"
DRIFT_LOG   = PROJECT_ROOT / "db"  / "schema_drift_log.json"
LOG_PATH    = PROJECT_ROOT / "db"  / "bootstrap_log.json"

# ---------------------------------------------------------------------------
# Landing gate check — do NOT proceed if landing zone is not validated
# ---------------------------------------------------------------------------
check_landing_gate(PROJECT_ROOT)

# ---------------------------------------------------------------------------
# Pipeline metadata
# ---------------------------------------------------------------------------
SOURCE_ID  = "pncp"
BATCH_ID   = str(uuid.uuid4())
STARTED_AT = datetime.now(timezone.utc).isoformat()

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"RAW_ROOT     : {RAW_ROOT}")
print(f"BRONZE_ROOT  : {BRONZE_ROOT}")
print(f"BATCH_ID     : {BATCH_ID}")
print(f"STARTED_AT   : {STARTED_AT}")


## Step 2 — Schema definition

In [ ]:
# ---------------------------------------------------------------------------
# Bronze schema — 55 source columns (all STRING) + 6 technical columns
# STRUCTs flattened with underscore separator (dot notation avoided in Parquet)
# Confirmed against EDA: 37 top-level columns stable across available months
# ---------------------------------------------------------------------------

# Scalar columns — direct from root level
SCALAR_COLUMNS = [
    # Identifiers
    "numeroControlePNCP",
    "numeroControlePncpCompra",
    "numeroContratoEmpenho",
    "sequencialContrato",
    "processo",
    "anoContrato",
    # Supplier
    "niFornecedor",
    "nomeRazaoSocialFornecedor",
    "tipoPessoa",
    "codigoPaisFornecedor",
    # Sub-contracted supplier (~100% null)
    "niFornecedorSubContratado",
    "nomeFornecedorSubContratado",
    "tipoPessoaSubContratada",
    # Contract content
    "objetoContrato",
    "informacaoComplementar",
    "receita",
    "numeroParcelas",
    "numeroRetificacao",
    # Dates — DuckDB infers DATE but Bronze stores STRING (ADR-002)
    "dataAssinatura",
    "dataVigenciaInicio",
    "dataVigenciaFim",
    "dataAtualizacao",
    "dataAtualizacaoGlobal",
    "dataPublicacaoPncp",
    # Values
    "valorInicial",
    "valorParcela",
    "valorGlobal",
    "valorAcumulado",
    # CIPI (97-100% null)
    "identificadorCipi",
    "urlCipi",
    # User
    "usuarioNome",
]

# Flattened STRUCT columns — format: {struct}_{field}
STRUCT_COLUMNS = [
    # orgaoEntidade — 0% null
    "orgaoEntidade_cnpj",
    "orgaoEntidade_razaoSocial",
    "orgaoEntidade_poderId",
    "orgaoEntidade_esferaId",
    # unidadeOrgao — 0% null
    "unidadeOrgao_ufNome",
    "unidadeOrgao_codigoUnidade",
    "unidadeOrgao_ufSigla",
    "unidadeOrgao_municipioNome",
    "unidadeOrgao_nomeUnidade",
    "unidadeOrgao_codigoIbge",
    # categoriaProcesso — 0% null
    "categoriaProcesso_id",
    "categoriaProcesso_nome",
    # tipoContrato — 0% null
    "tipoContrato_id",
    "tipoContrato_nome",
    # orgaoSubRogado — ~100% null, delegated contracts only
    "orgaoSubRogado_cnpj",
    "orgaoSubRogado_razaoSocial",
    "orgaoSubRogado_poderId",
    "orgaoSubRogado_esferaId",
    # unidadeSubRogada — ~100% null, delegated contracts only
    "unidadeSubRogada_ufNome",
    "unidadeSubRogada_codigoUnidade",
    "unidadeSubRogada_ufSigla",
    "unidadeSubRogada_municipioNome",
    "unidadeSubRogada_nomeUnidade",
    "unidadeSubRogada_codigoIbge",
]

# STRUCT_MAP — used in fast path SQL and slow path Python loop
STRUCT_MAP = {
    "orgaoEntidade"   : ["cnpj", "razaoSocial", "poderId", "esferaId"],
    "unidadeOrgao"    : ["ufNome", "codigoUnidade", "ufSigla", "municipioNome", "nomeUnidade", "codigoIbge"],
    "categoriaProcesso": ["id", "nome"],
    "tipoContrato"    : ["id", "nome"],
    "orgaoSubRogado"  : ["cnpj", "razaoSocial", "poderId", "esferaId"],
    "unidadeSubRogada": ["ufNome", "codigoUnidade", "ufSigla", "municipioNome", "nomeUnidade", "codigoIbge"],
}

BRONZE_COLUMNS   = SCALAR_COLUMNS + STRUCT_COLUMNS
TECHNICAL_COLUMNS = [
    "_rescued_data",   # JSON blob of unexpected source columns (None = no drift)
    "_source_id",
    "_batch_id",
    "_ingested_at",
    "_source_file",
    "_year_month",
]
ALL_COLUMNS = BRONZE_COLUMNS + TECHNICAL_COLUMNS

print(f"Scalar columns  : {len(SCALAR_COLUMNS)}")
print(f"Struct columns  : {len(STRUCT_COLUMNS)}")
print(f"Total source    : {len(BRONZE_COLUMNS)}")
print(f"Technical       : {len(TECHNICAL_COLUMNS)}")
print(f"Total columns   : {len(ALL_COLUMNS)}")


## Step 3 — Process months

In [ ]:
def _get_top_level_columns(source_path: str, con) -> set:
    """
    Return the set of top-level column names present in the source JSON.
    DuckDB infers the schema from the actual data.
    Used for schema drift detection before each write.
    """
    result = con.execute(
        f"DESCRIBE SELECT * FROM read_json_auto('{source_path}') LIMIT 0"
    ).fetchall()
    return {row[0] for row in result}


def _build_rescued_data(record: dict, known_scalar: set) -> str | None:
    """
    Build the _rescued_data JSON blob for a single record.

    Captures any top-level key not in known_scalar or the known struct roots.
    Returns None when no drift detected.

    Parameters
    ----------
    record       : single JSON record as a dict
    known_scalar : set of expected scalar column names
    """
    known_roots = known_scalar | set(STRUCT_MAP.keys())
    unexpected  = {k: v for k, v in record.items() if k not in known_roots}
    return json.dumps(unexpected, ensure_ascii=False, default=str) if unexpected else None


def _log_drift_event(year_month: str, rescued_count: int, sample: str) -> None:
    """
    Append a schema drift event to schema_drift_log.json.
    Local equivalent of INSERT INTO ops_quality_events in Databricks.
    """
    event = {
        "batch_id"      : BATCH_ID,
        "source_id"     : SOURCE_ID,
        "object"        : "bronze_pncp_contratos",
        "event_type"    : "SCHEMA_DRIFT",
        "severity"      : "WARN",
        "action"        : "CONTINUE",
        "year_month"    : year_month,
        "rescued_count" : rescued_count,
        "sample"        : sample[:300],
        "logged_at"     : datetime.now(timezone.utc).isoformat(),
    }
    existing = []
    if DRIFT_LOG.exists():
        with open(DRIFT_LOG, "r", encoding="utf-8") as f:
            try:
                existing = json.load(f)
            except json.JSONDecodeError:
                existing = []
    existing.append(event)
    DRIFT_LOG.parent.mkdir(parents=True, exist_ok=True)
    with open(DRIFT_LOG, "w", encoding="utf-8") as f:
        json.dump(existing, f, ensure_ascii=False, indent=2)


def _build_select_sql() -> str:
    """
    Build the SELECT SQL for Bronze transformation (fast path).

    - Scalar columns: CAST(col AS VARCHAR)
    - STRUCT subfields: CAST(struct."field" AS VARCHAR) AS struct_field
    - _rescued_data: NULL (only populated in slow path)

    Notes
    -----
    Field names are quoted to avoid conflicts with SQL reserved words.
    STRUCT_MAP is defined at module level — no need to pass as argument.
    """
    parts = []

    for col in SCALAR_COLUMNS:
        parts.append(f'CAST("{col}" AS VARCHAR) AS "{col}"')

    for struct_root, fields in STRUCT_MAP.items():
        for field in fields:
            col_name = f"{struct_root}_{field}"
            parts.append(
                f'CAST("{struct_root}"."{field}" AS VARCHAR) AS "{col_name}"'
            )

    parts.append("NULL::VARCHAR AS _rescued_data")
    return ",\n        ".join(parts)


def process_month(year_month: str, con) -> dict:
    """
    Read the raw PNCP JSON for a given month, flatten STRUCTs, apply Bronze schema,
    capture schema drift in _rescued_data, and write Parquet.

    Parameters
    ----------
    year_month : month string e.g. "2026-04"
    con        : DuckDB connection

    Returns
    -------
    dict with keys: year_month, status, records, rescued_count, file, error
    """
    source_file = RAW_ROOT / f"{year_month}.json"

    if not source_file.exists():
        return {
            "year_month"   : year_month,
            "status"       : "MISSING",
            "records"      : 0,
            "rescued_count": 0,
            "file"         : None,
            "error"        : f"File not found: {source_file}",
        }

    partition_dir = BRONZE_ROOT / f"_year_month={year_month}"
    partition_dir.mkdir(parents=True, exist_ok=True)
    output_file  = partition_dir / "data.parquet"
    source_path  = to_sql_path(source_file)
    output_path  = to_sql_path(output_file)

    try:
        # Schema drift detection
        known_roots = set(SCALAR_COLUMNS) | set(STRUCT_MAP.keys())
        source_cols = _get_top_level_columns(source_path, con)
        unexpected  = source_cols - known_roots

        # ---------------------------------------------------------------
        # Fast path — no drift, pure SQL
        # ---------------------------------------------------------------
        if not unexpected:
            select_sql = _build_select_sql()
            con.execute(f"""
                COPY (
                    SELECT
                        {select_sql},
                        '{SOURCE_ID}'     AS _source_id,
                        '{BATCH_ID}'      AS _batch_id,
                        CURRENT_TIMESTAMP AS _ingested_at,
                        '{source_path}'  AS _source_file,
                        '{year_month}'   AS _year_month
                    FROM read_json_auto('{source_path}')
                ) TO '{output_path}'
                (FORMAT PARQUET, COMPRESSION SNAPPY)
            """)

            records = con.execute(
                f"SELECT COUNT(*) FROM read_parquet('{output_path}')"
            ).fetchone()[0]

            return {
                "year_month"   : year_month,
                "status"       : "SUCCESS",
                "records"      : records,
                "rescued_count": 0,
                "file"         : str(output_file),
                "error"        : None,
            }

        # ---------------------------------------------------------------
        # Slow path — drift detected, Python per-record processing
        # ---------------------------------------------------------------
        with open(source_file, "r", encoding="utf-8") as f:
            raw_data = json.load(f)

        enriched_records = []
        rescued_count    = 0
        sample_rescued   = None

        for record in raw_data:
            rescued = _build_rescued_data(record, set(SCALAR_COLUMNS))
            if rescued:
                rescued_count += 1
                if sample_rescued is None:
                    sample_rescued = rescued

            row = {}
            for col in SCALAR_COLUMNS:
                val = record.get(col)
                row[col] = None if val is None else str(val)

            for struct_root, fields in STRUCT_MAP.items():
                struct_val = record.get(struct_root) or {}
                for field in fields:
                    col_name = f"{struct_root}_{field}"
                    val = struct_val.get(field) if isinstance(struct_val, dict) else None
                    row[col_name] = None if val is None else str(val)

            row["_rescued_data"] = rescued
            row["_source_id"]    = SOURCE_ID
            row["_batch_id"]     = BATCH_ID
            row["_ingested_at"]  = datetime.now(timezone.utc).isoformat()
            row["_source_file"]  = source_path
            row["_year_month"]   = year_month
            enriched_records.append(row)

        con.execute("CREATE OR REPLACE TEMP TABLE _drift_staging AS "
                    "SELECT * FROM enriched_records")
        con.execute(f"""
            COPY (SELECT * FROM _drift_staging)
            TO '{output_path}'
            (FORMAT PARQUET, COMPRESSION SNAPPY)
        """)

        records = con.execute(
            f"SELECT COUNT(*) FROM read_parquet('{output_path}')"
        ).fetchone()[0]

        if rescued_count > 0:
            _log_drift_event(year_month, rescued_count, sample_rescued or "")

        return {
            "year_month"   : year_month,
            "status"       : "SUCCESS",
            "records"      : records,
            "rescued_count": rescued_count,
            "file"         : str(output_file),
            "error"        : None,
        }

    except Exception as e:
        return {
            "year_month"   : year_month,
            "status"       : "ERROR",
            "records"      : 0,
            "rescued_count": 0,
            "file"         : None,
            "error"        : str(e),
        }


# ---------------------------------------------------------------------------
# Execute — process all available months
# ---------------------------------------------------------------------------
available_months = sorted([f.stem for f in RAW_ROOT.glob("*.json")])

if not available_months:
    raise FileNotFoundError(
        f"No JSON files found in {RAW_ROOT}\n"
        "Run 02_bootstrap_pncp.py first."
    )

print(f"Months available : {len(available_months)}")
print(f"Range            : {available_months[0]} -> {available_months[-1]}")
print()

BRONZE_ROOT.mkdir(parents=True, exist_ok=True)
results = []
con = get_connection()  # in-memory — Parquet I/O does not require persistent DB

for i, ym in enumerate(available_months, 1):
    res = process_month(ym, con)
    results.append(res)
    icon         = "OK " if res["status"] == "SUCCESS" else "ERR"
    drift_suffix = f" — {res['rescued_count']:,} rescued" if res.get("rescued_count", 0) > 0 else ""
    print(f"  [{icon}] [{i:02d}/{len(available_months)}] {ym} "
          f"— {res['status']} — {res['records']:>8,} records{drift_suffix}")

con.close()

total_records = sum(r["records"] for r in results)
success_count = sum(1 for r in results if r["status"] == "SUCCESS")
error_count   = sum(1 for r in results if r["status"] == "ERROR")
missing_count = sum(1 for r in results if r["status"] == "MISSING")
drift_months  = [r["year_month"] for r in results if r.get("rescued_count", 0) > 0]
total_rescued = sum(r.get("rescued_count", 0) for r in results)

print()
print(f"  Success  : {success_count}")
print(f"  Error    : {error_count}")
print(f"  Missing  : {missing_count}")
print(f"  Records  : {total_records:,}")
print(f"  Drift    : {len(drift_months)} month(s) — {total_rescued:,} rescued records")
if drift_months:
    print(f"  Months   : {drift_months}")
if error_count:
    print()
    for r in results:
        if r["status"] == "ERROR":
            print(f"  ERR {r['year_month']} — {r['error']}")


## Step 4 — Validation

In [ ]:
suite       = CheckSuite("bronze_pncp_contratos")
con_val     = get_connection()
bronze_glob = to_sql_path(BRONZE_ROOT / "**" / "*.parquet")

# Check 1 — no processing errors or missing files
suite.add(
    "No processing errors or missing files",
    error_count == 0 and missing_count == 0,
    f"errors={error_count}, missing={missing_count}",
)

# Check 2 — all partition Parquet files exist
missing_parts = [
    ym for ym in available_months
    if not (BRONZE_ROOT / f"_year_month={ym}" / "data.parquet").exists()
]
suite.add(
    "All partition Parquet files exist",
    not missing_parts,
    f"missing: {missing_parts}" if missing_parts else "all present",
)

# Check 3 — no empty partitions
empty_parts = [r["year_month"] for r in results if r["status"] == "SUCCESS" and r["records"] == 0]
suite.add(
    "No empty partitions",
    not empty_parts,
    f"empty: {empty_parts}" if empty_parts else "all non-empty",
)

# Check 4 — total record count
bronze_total = con_val.execute(
    f"SELECT COUNT(*) FROM read_parquet('{bronze_glob}', hive_partitioning=true)"
).fetchone()[0]
suite.add(
    "Record count matches source",
    total_records == bronze_total,
    f"source={total_records:,}  bronze={bronze_total:,}",
)

# Check 5 — column count
actual_cols = con_val.execute(f"""
    SELECT COUNT(*) FROM (
        DESCRIBE SELECT * FROM read_parquet('{bronze_glob}', hive_partitioning=true)
        LIMIT 0
    )
""").fetchone()[0]
suite.add(
    "Column count correct",
    actual_cols == len(ALL_COLUMNS),
    f"found={actual_cols}  expected={len(ALL_COLUMNS)}",
)

# Check 6 — all technical columns present
existing_cols = {
    row[0] for row in con_val.execute(
        f"DESCRIBE SELECT * FROM read_parquet('{bronze_glob}', hive_partitioning=true) LIMIT 0"
    ).fetchall()
}
missing_tech = [c for c in TECHNICAL_COLUMNS if c not in existing_cols]
suite.add(
    "All technical columns present",
    not missing_tech,
    f"missing: {missing_tech}" if missing_tech else "all present",
)

# Check 7 — key struct columns present (flattening worked)
key_struct = ["orgaoEntidade_cnpj", "unidadeOrgao_ufSigla",
              "categoriaProcesso_nome", "tipoContrato_nome"]
missing_struct = [c for c in key_struct if c not in existing_cols]
suite.add(
    "Key STRUCT columns present (flattening OK)",
    not missing_struct,
    f"missing: {missing_struct}" if missing_struct else "all present",
)

# Check 8 — niFornecedor not excessively null
null_ni = con_val.execute(
    f"SELECT COUNT(*) FROM read_parquet('{bronze_glob}', hive_partitioning=true)"
    " WHERE niFornecedor IS NULL"
).fetchone()[0]
suite.add(
    "niFornecedor null rate acceptable",
    null_ni / bronze_total < 0.05 if bronze_total else True,
    f"{null_ni:,} nulls ({null_ni/bronze_total*100:.2f}%)" if bronze_total else "n/a",
)

# Check 9 — schema drift monitored (always PASS — drift is a warning, not a blocker)
suite.add(
    "Schema drift monitored",
    True,
    f"{len(drift_months)} month(s) with rescued data"
    + (f": {drift_months}" if drift_months else " — none detected"),
)

con_val.close()  # close before potential raise
suite.report()
suite.assert_all_pass()


## Step 5 — Ops registration

In [ ]:
files_written = sum(1 for r in results if r["status"] == "SUCCESS")
bytes_written = sum(
    (BRONZE_ROOT / f"_year_month={r['year_month']}" / "data.parquet").stat().st_size
    for r in results if r["status"] == "SUCCESS"
)
period       = f"{available_months[0]}/{available_months[-1]}"
batch_status = "SUCCESS" if error_count == 0 else "PARTIAL"
errors_detail = (
    "; ".join(f"{r['year_month']}: {r['error'][:60]}" for r in results if r["status"] == "ERROR")
    if error_count > 0 else None
)

entry = make_entry(
    source_id    = SOURCE_ID,
    period       = period,
    status       = batch_status,
    records      = total_records,
    bytes_written = bytes_written,
    started_at   = STARTED_AT,
    finished_at  = datetime.now(timezone.utc).isoformat(),
    error_message = errors_detail,
    # Bronze-specific extra fields
    batch_id         = BATCH_ID,
    layer            = "bronze",
    object           = "pncp_contratos",
    files            = files_written,
    has_rescued_data = len(drift_months) > 0,
    drift_months     = len(drift_months),
)

append_log(LOG_PATH, entry)

print(f"Batch registered   : {BATCH_ID}")
print(f"Status             : {batch_status}")
print(f"Records            : {total_records:,}")
print(f"Files              : {files_written}")
print(f"Bytes written      : {bytes_written / (1024*1024):.1f} MB")
print(f"Period             : {period}")
print(f"Has rescued data   : {len(drift_months) > 0}")
print(f"Log                : {LOG_PATH}")
